# Section 03: Storing the Data

**CareerAlign GCP Professional Data Engineer**

This notebook covers:
1. Storage service selection decision tree
2. BigQuery partitioning, clustering, and optimization
3. Bigtable row key design and anti-patterns
4. Cloud Storage lifecycle policies
5. Data warehouse modeling patterns (denormalization, nested fields)
6. File format comparison (Parquet, Avro, ORC, CSV, JSON)

> **Note**: Cells marked with `# REQUIRES GCP` need a GCP project. Local-only cells run anywhere.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/gcp-pde/notebooks/03-storing-the-data.ipynb)

---
## Setup & Installations

In [ ]:
# Install required packages
!pip install -q google-cloud-bigquery google-cloud-bigtable google-cloud-storage pandas numpy pyarrow

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

print("Setup complete!")

---
## 1. Storage Service Selection

A decision tree for choosing the right GCP storage service.

In [ ]:
# --- Storage Service Comparison ---

storage_services = pd.DataFrame({
    'Service': ['BigQuery', 'Cloud Spanner', 'AlloyDB', 'Cloud SQL',
                'Bigtable', 'Firestore', 'Memorystore', 'Cloud Storage'],
    'Type': ['Data Warehouse', 'Relational (Global)', 'Relational (Regional)',
             'Relational (Regional)', 'NoSQL (Wide-column)', 'NoSQL (Document)',
             'In-memory (Cache)', 'Object Storage'],
    'Scale': ['Petabytes', 'Unlimited', '64 TB+', '64 TB',
              'Petabytes', 'Terabytes', 'Gigabytes', 'Exabytes'],
    'Latency': ['Seconds', 'Single-digit ms', 'Low ms', 'Low ms',
                'Single-digit ms', 'Low ms', 'Sub-ms', 'Varies'],
    'ACID': ['Per-DML + multi-stmt', 'Full (global)', 'Full', 'Full',
             'Row-level only', 'Full (500 docs)', 'No', 'Object-level'],
    'SLA': ['99.99%', '99.999%', '99.99%', '99.95%',
            '99.99%', '99.999%', '99.9%', '99.95-99.99%'],
    'Best For': [
        'SQL analytics, ML, BI',
        'Global ACID, financial',
        'PG + fast analytics',
        'Simple web/app OLTP',
        'Time-series, IoT, high throughput',
        'Mobile/web real-time',
        'Caching, sessions',
        'Files, data lake, backups'
    ]
})

print("=== GCP Storage Services Comparison ===")
print(storage_services.to_string(index=False))

In [ ]:
# --- Decision Tree Scenarios ---

scenarios = [
    ('Petabyte-scale SQL analytics', 'BigQuery'),
    ('Global financial transactions with ACID', 'Cloud Spanner'),
    ('PostgreSQL with analytical performance', 'AlloyDB'),
    ('Simple MySQL web backend', 'Cloud SQL'),
    ('IoT time-series at 1M writes/sec', 'Bigtable'),
    ('Mobile app with offline sync', 'Firestore'),
    ('Session cache for web app', 'Memorystore (Redis)'),
    ('Store ML training data files', 'Cloud Storage'),
    ('Real-time gaming leaderboard', 'Memorystore (Redis)'),
    ('HBase migration from Hadoop', 'Bigtable'),
    ('Graph-like queries on documents', 'Firestore'),
    ('Data lake with governance', 'Cloud Storage + BigLake + Dataplex'),
]

print("=== Storage Selection Scenarios ===")
for scenario, answer in scenarios:
    print(f"  Q: {scenario}")
    print(f"  A: {answer}")
    print()

---
## 2. BigQuery Partitioning & Clustering

Demonstrate cost savings from partitioning and clustering with simulated data.

In [ ]:
# --- Simulate partition pruning cost savings ---

np.random.seed(42)
n = 365 * 10000  # 10K events per day for 1 year

dates = np.random.choice(pd.date_range('2024-01-01', '2024-12-31'), n)
regions = np.random.choice(['US', 'EU', 'APAC', 'LATAM'], n, p=[0.4, 0.3, 0.2, 0.1])
amounts = np.random.lognormal(3, 1, n).round(2)

df = pd.DataFrame({'event_date': dates, 'region': regions, 'amount': amounts})

# Simulate bytes per row
bytes_per_row = 200
total_bytes = len(df) * bytes_per_row
total_gb = total_bytes / 1e9
cost_per_tb = 6.25

# Full table scan
full_scan_cost = (total_gb / 1000) * cost_per_tb

# Partition pruning (query 1 month)
month_rows = df[df['event_date'].dt.month == 3]
month_gb = (len(month_rows) * bytes_per_row) / 1e9
month_cost = (month_gb / 1000) * cost_per_tb

# Partition + cluster pruning (1 month + 1 region)
filtered = df[(df['event_date'].dt.month == 3) & (df['region'] == 'US')]
filtered_gb = (len(filtered) * bytes_per_row) / 1e9
filtered_cost = (filtered_gb / 1000) * cost_per_tb

print("=== BigQuery Cost Simulation ===")
print(f"Total dataset: {len(df):,} rows, ~{total_gb:.2f} GB")
print(f"{'Query Type':<35} {'Rows Scanned':>15} {'GB Scanned':>12} {'Cost':>10} {'Savings':>10}")
print("-" * 85)
print(f"{'Full table scan (SELECT *)':<35} {len(df):>15,} {total_gb:>12.2f} ${full_scan_cost:>9.4f} {'0%':>10}")
print(f"{'Partition pruned (1 month)':<35} {len(month_rows):>15,} {month_gb:>12.2f} ${month_cost:>9.4f} {f'{(1-month_cost/full_scan_cost)*100:.0f}%':>10}")
print(f"{'Partition + cluster (month+region)':<35} {len(filtered):>15,} {filtered_gb:>12.2f} ${filtered_cost:>9.4f} {f'{(1-filtered_cost/full_scan_cost)*100:.0f}%':>10}")

In [ ]:
# --- BigQuery DDL Examples ---

bq_ddl = """
-- Create partitioned + clustered table
CREATE TABLE `project.dataset.events`
(
  event_id STRING,
  user_id STRING,
  event_type STRING,
  amount NUMERIC,
  region STRING,
  event_time TIMESTAMP
)
PARTITION BY DATE(event_time)
CLUSTER BY user_id, region
OPTIONS (
  partition_expiration_days = 365,
  require_partition_filter = true,  -- Prevent accidental full scans
  description = "Production events table"
);

-- GOOD query: uses partition + cluster pruning
SELECT user_id, SUM(amount)
FROM `project.dataset.events`
WHERE DATE(event_time) = '2025-03-01'
  AND region = 'US'
GROUP BY user_id;

-- BAD query: function on partition column prevents pruning
SELECT * FROM `project.dataset.events`
WHERE EXTRACT(YEAR FROM event_time) = 2025;  -- FULL TABLE SCAN!

-- Materialized view with staleness
CREATE MATERIALIZED VIEW `project.dataset.daily_summary`
PARTITION BY event_date
OPTIONS (
  enable_refresh = true,
  refresh_interval_minutes = 30,
  max_staleness = INTERVAL "1:0:0" HOUR TO SECOND
)
AS SELECT
  DATE(event_time) AS event_date,
  region,
  COUNT(*) AS event_count,
  SUM(amount) AS total_amount
FROM `project.dataset.events`
GROUP BY 1, 2;
"""

print("=== BigQuery DDL: Partitioning, Clustering, Materialized Views ===")
print(bq_ddl)

---
## 3. Bigtable Row Key Design

Demonstrate good and bad row key patterns for Bigtable.

In [ ]:
# --- Bigtable Row Key Design Patterns ---

# Simulate writes and show distribution
import hashlib

devices = [f'sensor_{i:04d}' for i in range(1, 101)]
timestamps = pd.date_range('2025-03-01', periods=1000, freq='s')

# BAD: Timestamp-first row key (hotspotting)
bad_keys = [f"{ts.strftime('%Y%m%d%H%M%S')}#{np.random.choice(devices)}" 
            for ts in timestamps]

# GOOD: Device-first with reversed timestamp
max_ts = 9999999999
good_keys = [f"{np.random.choice(devices)}#{max_ts - int(ts.timestamp())}"
             for ts in timestamps]

# Show key distribution (first character = which tablet gets the write)
bad_prefixes = pd.Series([k[:10] for k in bad_keys]).value_counts().head(5)
good_prefixes = pd.Series([k.split('#')[0] for k in good_keys]).value_counts().head(10)

print("=== Bigtable Row Key Design ===")
print()
print("BAD PATTERN: timestamp#device_id")
print("  All writes go to same tablet (latest timestamp prefix)!")
print(f"  Top 5 prefixes (showing hotspot):")
for prefix, count in bad_prefixes.items():
    print(f"    {prefix}... -> {count} writes")

print()
print("GOOD PATTERN: device_id#reversed_timestamp")
print("  Writes distribute across tablets (different device prefixes)")
print(f"  Top 10 device prefixes (even distribution):")
for prefix, count in good_prefixes.items():
    print(f"    {prefix} -> {count} writes")

print()
print("Rules for row key design:")
print("  1. Avoid monotonically increasing keys (timestamps, sequence IDs)")
print("  2. Put high-cardinality, well-distributed field first")
print("  3. Reverse timestamp for latest-first scans")
print("  4. Consider salting with hash prefix for extreme write throughput")

---
## 4. Cloud Storage Lifecycle Policies

Show lifecycle configuration for automatic storage class transitions.

In [ ]:
# --- Storage Class Cost Comparison ---

storage_classes = pd.DataFrame({
    'Class': ['Standard', 'Nearline', 'Coldline', 'Archive'],
    'Storage $/GB/mo': [0.020, 0.010, 0.004, 0.0012],
    'Retrieval $/GB': [0.00, 0.01, 0.02, 0.05],
    'Min Duration': ['None', '30 days', '90 days', '365 days'],
    'Best For': [
        'Frequently accessed, active data',
        'Monthly access: backups, logs',
        'Quarterly access: DR, compliance',
        'Yearly access: legal, audit'
    ]
})

# Calculate costs for 100 TB over 1 year
tb = 100
gb = tb * 1024
months = 12

print("=== Storage Class Comparison (100 TB for 12 months) ===")
print(storage_classes.to_string(index=False))
print()
for _, row in storage_classes.iterrows():
    annual_cost = gb * row['Storage $/GB/mo'] * months
    print(f"  {row['Class']}: ${annual_cost:,.0f}/year storage")

In [ ]:
# --- Lifecycle Policy JSON ---

lifecycle_json = {
    "rule": [
        {
            "action": {"type": "SetStorageClass", "storageClass": "NEARLINE"},
            "condition": {"age": 30, "matchesStorageClass": ["STANDARD"]}
        },
        {
            "action": {"type": "SetStorageClass", "storageClass": "COLDLINE"},
            "condition": {"age": 90, "matchesStorageClass": ["NEARLINE"]}
        },
        {
            "action": {"type": "SetStorageClass", "storageClass": "ARCHIVE"},
            "condition": {"age": 365, "matchesStorageClass": ["COLDLINE"]}
        },
        {
            "action": {"type": "Delete"},
            "condition": {"age": 730}
        }
    ]
}

print("=== Cloud Storage Lifecycle Policy ===")
print(json.dumps(lifecycle_json, indent=2))
print()
print("Timeline:")
print("  Day 0-29:    Standard")
print("  Day 30-89:   Nearline")
print("  Day 90-364:  Coldline")
print("  Day 365-729: Archive")
print("  Day 730+:    Deleted")
print()
print("# Apply with: gsutil lifecycle set lifecycle.json gs://my-bucket")

---
## 5. Data Warehouse Modeling: Denormalization

Show BigQuery nested/repeated fields and denormalization patterns.

In [ ]:
# --- Denormalization with nested fields ---

nested_sql = """
-- Create denormalized table with STRUCT and ARRAY
CREATE TABLE `project.dataset.orders_denormalized` AS
SELECT
  o.order_id,
  o.order_date,
  STRUCT(
    c.customer_id,
    c.name,
    c.email,
    c.tier
  ) AS customer,
  STRUCT(
    a.street,
    a.city,
    a.state,
    a.country
  ) AS shipping_address,
  ARRAY_AGG(STRUCT(
    oi.product_id,
    oi.product_name,
    oi.quantity,
    oi.unit_price,
    oi.quantity * oi.unit_price AS line_total
  )) AS line_items,
  SUM(oi.quantity * oi.unit_price) AS order_total
FROM orders o
JOIN customers c USING (customer_id)
JOIN addresses a ON o.shipping_address_id = a.address_id
JOIN order_items oi USING (order_id)
GROUP BY 1, 2, 3, 4;

-- Query the denormalized table (no JOINs needed!)
SELECT
  order_id,
  customer.name,
  customer.tier,
  shipping_address.city,
  order_total,
  ARRAY_LENGTH(line_items) AS num_items,
  (SELECT MAX(li.unit_price) FROM UNNEST(line_items) li) AS max_item_price
FROM `project.dataset.orders_denormalized`
WHERE order_date >= '2025-01-01'
  AND customer.tier = 'gold'
ORDER BY order_total DESC
LIMIT 100;
"""

print("=== BigQuery Denormalization with STRUCT/ARRAY ===")
print(nested_sql)
print("Benefits:")
print("  - No JOINs at query time -> faster queries, lower slot usage")
print("  - STRUCT: nested single record (1:1)")
print("  - ARRAY: nested repeated records (1:N)")
print("  - UNNEST: flatten arrays for aggregation")

---
## 6. File Format Comparison

Compare Parquet, Avro, ORC, CSV, and JSON using actual file sizes.

In [ ]:
import os
import tempfile

# Create sample data
np.random.seed(42)
n = 10000

df_format = pd.DataFrame({
    'user_id': [f'user_{i:06d}' for i in range(n)],
    'event_time': pd.date_range('2025-01-01', periods=n, freq='min'),
    'amount': np.random.lognormal(3, 1, n).round(2),
    'region': np.random.choice(['US', 'EU', 'APAC', 'LATAM'], n),
    'device': np.random.choice(['mobile', 'desktop', 'tablet'], n),
    'event_type': np.random.choice(['purchase', 'browse', 'click', 'signup'], n),
})

tmpdir = tempfile.mkdtemp()

# Save in different formats
csv_path = os.path.join(tmpdir, 'data.csv')
json_path = os.path.join(tmpdir, 'data.json')
parquet_path = os.path.join(tmpdir, 'data.parquet')

df_format.to_csv(csv_path, index=False)
df_format.to_json(json_path, orient='records', lines=True)
df_format.to_parquet(parquet_path, index=False, compression='snappy')

sizes = {
    'CSV': os.path.getsize(csv_path),
    'JSON (NDJSON)': os.path.getsize(json_path),
    'Parquet (Snappy)': os.path.getsize(parquet_path),
}

print(f"=== File Format Size Comparison ({n:,} rows) ===")
print(f"{'Format':<20} {'Size (KB)':>12} {'Ratio':>10}")
print('-' * 45)
csv_size = sizes['CSV']
for fmt, size in sizes.items():
    ratio = size / csv_size
    print(f"{fmt:<20} {size/1024:>12.1f} {ratio:>10.2f}x")

print()
print("Key takeaways:")
print("  - Parquet is 3-5x smaller than CSV (columnar + compression)")
print("  - JSON is larger than CSV (field names repeated per record)")
print("  - For data lakes: ALWAYS use Parquet")
print("  - For streaming/CDC: Avro (schema evolution, compact)")
print("  - For import/export: CSV/JSON (human readable)")

---
## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **Storage Selection** | BQ=analytics, Spanner=global ACID, AlloyDB=PG+analytics, Bigtable=time-series, Firestore=mobile/web |
| **BigQuery Optimization** | Partition by date, cluster by filter columns, require_partition_filter, materialized views |
| **Bigtable Row Keys** | Never start with timestamp. Use device_id#reversed_timestamp. Avoid hotspotting. |
| **Cloud Storage** | Lifecycle policies for automatic class transitions. Standard->Nearline->Coldline->Archive->Delete |
| **Denormalization** | Use STRUCT (1:1) and ARRAY (1:N) for denormalized tables. Avoids JOINs. |
| **File Formats** | Parquet for analytics (columnar, compressed). Avro for streaming (row-based, schema evolution). |